# exp4 R1 8 blind crossdataset

**Self-contained Colab notebook.** Run the setup cells once (top to bottom), then the experiment. The four shared modules are written to `/content` automatically, so you do NOT need to upload a `shared/` folder. Edit Drive paths only in the `config_paths.py` cell below.

> Runtime: set **Runtime -> Change runtime type -> GPU** for the inference and training experiments.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
print('Drive mounted; /content on path.')

In [ ]:
# scikit-image backs metrics.calc_ssim; guarantee it for every notebook
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-image'])
print('scikit-image ready')

In [ ]:
%%writefile /content/config_paths.py
"""
Central path + constant configuration for all JBHI revision Colab experiments.

Derived verbatim from the author's existing notebooks:
  - COLAB_IEEE_FIGURES.ipynb      (paper baselines + EPHNOGRAM chunks + split)
  - cross_dataset_validation.ipynb (SensSmart-Tech + Trail-4 generator)

Every experiment script imports from this file so paths live in ONE place.
If your Drive layout differs, edit ONLY this file.
"""

import os
import torch

# ============================================================================
# >>> EDIT THIS BLOCK IF YOUR DRIVE MOVES <<<
# Updated 2026-07-14: the dataset is now a SHARED folder mounted at
# /content/drive/MyDrive/JBHI-Revision/October Thesis Dataset. If it moves again,
# change ONLY DRIVE_ROOT and OCT_DIR. NOTE: DATA is READ from OCT_DIR; everything
# the code WRITES (retrained checkpoints, outputs) goes under your own Drive
# (OUT_ROOT), so a read-only shared dataset still works.
# ============================================================================
DRIVE_ROOT = "/content/drive/MyDrive"
# Shared dataset root: contains "Refine-Chunked", "Best-epoch-trail-4", etc. (read-only OK).
OCT_DIR    = f"{DRIVE_ROOT}/JBHI-Revision/October Thesis Dataset"

# ----------------------------------------------------------------------------
# EPHNOGRAM data (training-domain test set = 453 segments, 3 subjects)
# .npz keys per chunk: 'ecg', 'pcg', 'subject_id', 'scenario'
#   ecg, pcg : float32 arrays, 64000 samples (8 s @ 8 kHz), RAW (not normalized)
#   subject_id : e.g. 'S009', 'S019', 'S023'
#   scenario   : free-text, e.g. 'rest', 'exercise bicycle stress test'
# ----------------------------------------------------------------------------
EPH_CHUNKS_DIR = f"{OCT_DIR}/Refine-Chunked"
# Subject split. keys: train/val/test_subjects. IF THIS FILE IS ABSENT the
# notebooks fall back to an arbitrary split, which gives the WRONG test set
# (the 2026-07-10 run hit this: 22 segments / 4 subjects instead of the paper's
# 453 segments / 3 subjects). Upload the real split JSON, or point this at it.
SPLIT_FILE     = f"{EPH_CHUNKS_DIR}/train_val_test_split.json"

# ----------------------------------------------------------------------------
# Model checkpoints  --  CONFIRM ALL FOUR EXIST IN THE NEW ACCOUNT
#
#   "Proposed" MUST be the Trail-4 generator (polarity fix, BatchNorm output),
#   i.e. trail4_best.pt or trail4_generator.pth. A "trial2_*" / Trail-2 file is
#   the WRONG (Tanh) model: it loads only partially under strict=False and
#   produces a near-flat, invalid reconstruction (see the 2026-07-10 run).
#
#   The three baselines lived under a SEPARATE top-level folder
#   (Paper1_ECG_Reconstruction) in the old account. Confirm they were migrated;
#   if they now sit under OCT_DIR, repoint them below.
# ----------------------------------------------------------------------------
CKPT = {
    # Baselines are RETRAINED by exp7 and SAVED to your own writable Drive
    # (DRIVE_ROOT/JBHI_Revision/baseline_checkpoints == OUT_ROOT/baseline_checkpoints),
    # so a read-only shared dataset is fine. Same paths are used to load them later.
    "CNN":      f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/CNN_EncoderDecoder/best_model.pt",
    "UNet":     f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/UNet_Baseline/best_model.pt",
    "WGAN-GP":  f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/WGAN_GP/best_model.pt",
    "Proposed": f"{OCT_DIR}/Best-epoch-trail-4/trail4_best.pt",   # read from the shared dataset
}
CKPT_PROPOSED_ALT = f"{OCT_DIR}/trail4_generator.pth"   # fallback copy (state_dict)

# ----------------------------------------------------------------------------
# SensSmart-Tech cross-dataset (32 subjects, 338 records @ 1 kHz -> 8 kHz)
#   WFDB pairs: {base}_ecg.hea/.dat and {base}_pcg.hea/.dat ; ECG lead II
# ----------------------------------------------------------------------------
SENSSMART_WFDB  = f"{OCT_DIR}/senssmarttech-database/WFDB"
SENSSMART_PREP  = f"{OCT_DIR}/milestone2_preprocessed"     # cached .npz (keys ecg/pcg)

# ----------------------------------------------------------------------------
# Output root for ALL revision artifacts (created on demand)
#   Each experiment writes to a named sub-folder here.
# ----------------------------------------------------------------------------
OUT_ROOT = f"{DRIVE_ROOT}/JBHI_Revision"

def out_dir(name):
    """Return (and create) an output sub-folder under OUT_ROOT."""
    p = os.path.join(OUT_ROOT, name)
    os.makedirs(p, exist_ok=True)
    return p

# ----------------------------------------------------------------------------
# Signal + inference constants
# ----------------------------------------------------------------------------
FS          = 8000        # sampling rate (Hz)
SEG_LEN     = 64000       # samples per segment (8 s)
NUM_SCENARIOS = 3         # rest / exercise / recovery one-hot
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# R-peak detector settings (identical to metrics.ipynb; keep fixed for reproducibility)
RPEAK_DISTANCE   = 400    # min inter-peak samples = 50 ms @ 8 kHz
RPEAK_PROMINENCE = 0.5

# SensSmart preprocessing bands (from cross_dataset_validation.ipynb)
ECG_BAND = (0.5, 40.0)
PCG_BAND = (25.0, 400.0)
FILTER_ORDER = 4

# Scenario one-hots
SCEN_REST     = [1.0, 0.0, 0.0]
SCEN_EXERCISE = [0.0, 1.0, 0.0]
SCEN_RECOVERY = [0.0, 0.0, 1.0]


def scenario_to_index(scenario_str):
    """Map EPHNOGRAM free-text scenario -> class index {0,1,2}.

    Matches the mapping used in COLAB_IEEE_FIGURES.ipynb.
    """
    s = str(scenario_str).lower()
    if "rest" in s or "laying" in s:
        return 0
    if "exercise" in s or "bicycle" in s or "treadmill" in s or "running" in s:
        return 1
    return 2

In [ ]:
%%writefile /content/models.py
"""
Model architectures for the JBHI revision experiments.

Reproduced VERBATIM from the author's COLAB_IEEE_FIGURES.ipynb (paper Table II
baselines) and models.ipynb (ablation trails), so the checkpoints in
config_paths.CKPT load cleanly.

Which class loads which checkpoint
----------------------------------
    CKPT['CNN']      -> CNNEncoderDecoder()        # no scenario, forward(x)
    CKPT['UNet']     -> UNetGenerator()            # pix2pix, forward(x)
    CKPT['WGAN-GP']  -> UNetGenerator()            # same generator as U-Net
    CKPT['Proposed'] -> ImprovedGeneratorNoTanh()  # scenario, forward(x, s_onehot)

The proposed model's forward takes a scenario one-hot; the three baselines take
only the PCG. Every inference script must branch on this (helper `run_generator`
below does it for you).

For the R1.4 component ablation, use `ConfigurableGenerator`, which exposes the
three grouped factors as switches: use_attention, use_dilated, and
output_activation in {'batchnorm','tanh','linear'}.

NOTE ON PARAM COUNTS: the submitted paper's numbers (CNN 9.8M, U-Net 13.6M,
Proposed 14.0M) do NOT match these classes (which give ~larger counts). Always
report count_params(model_loaded_from_checkpoint) computed here -- see R2.2/R2.5.
"""

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ===========================================================================
# Shared bottleneck modules
# ===========================================================================
class MultiHeadSelfAttention(nn.Module):
    """8-head self-attention over the temporal axis at the bottleneck."""
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        assert channels % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1, bias=False)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, C, T = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, T)
        qkv = qkv.permute(1, 0, 2, 4, 3)          # (3, B, heads, T, head_dim)
        # .contiguous() is REQUIRED for SDPA to select the flash / memory-efficient
        # kernel (O(T) memory). Without it the permuted views fall back to the MATH
        # kernel, which materializes the T x T score matrix -- that OOM-ed the
        # attention variants even on a 40 GB A100 at batch 48. Mathematically
        # identical to softmax(QK^T/sqrt(d))V, so existing checkpoints are unaffected.
        q, k, v = qkv[0].contiguous(), qkv[1].contiguous(), qkv[2].contiguous()
        p = self.dropout.p if self.training else 0.0
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=p)
        out = out.transpose(2, 3).reshape(B, C, T)
        return self.proj(out) + x


class MultiScaleDilatedBlock(nn.Module):
    """Parallel dilated convs at rates 1, 2, 4 with a 1x1 fusion + residual."""
    def __init__(self, channels):
        super().__init__()
        c1 = channels // 3 + (1 if channels % 3 >= 1 else 0)
        c2 = channels // 3 + (1 if channels % 3 >= 2 else 0)
        c3 = channels // 3
        self.conv_d1 = nn.Conv1d(channels, c1, 3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(channels, c2, 3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(channels, c3, 3, padding=4, dilation=4)
        self.fusion = nn.Sequential(
            nn.Conv1d(channels, channels, 1),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True))

    def forward(self, x):
        cat = torch.cat([self.conv_d1(x), self.conv_d2(x), self.conv_d4(x)], dim=1)
        return self.fusion(cat) + x


class EnhancedResidualBlock(nn.Module):
    """Residual block with optional MHSA and MSDC (Trails 2-4)."""
    def __init__(self, channels, use_attention=False, use_dilated=True):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.use_dilated = use_dilated
        if use_dilated:
            self.dilated_block = MultiScaleDilatedBlock(channels)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.use_attention = use_attention
        if use_attention:
            self.attention = MultiHeadSelfAttention(channels, num_heads=8)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        if self.use_dilated:
            out = self.dilated_block(out)
        out = self.bn2(self.conv2(out))
        if self.use_attention:
            out = self.attention(out)
        return self.relu(out + residual)


# ===========================================================================
# Paper Table II baseline: CNN Encoder-Decoder (8-level, no skips, no scenario)
# ===========================================================================
class ConvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.block(x)


class DeconvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)


class CNNEncoderDecoder(nn.Module):
    """Paper CNN baseline -> loads CKPT['CNN']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, base_filters=64):
        super().__init__()
        bf = base_filters
        self.enc1 = ConvBlock(in_channels, bf)
        self.enc2 = ConvBlock(bf, bf * 2)
        self.enc3 = ConvBlock(bf * 2, bf * 4)
        self.enc4 = ConvBlock(bf * 4, bf * 8)
        self.enc5 = ConvBlock(bf * 8, bf * 8)
        self.enc6 = ConvBlock(bf * 8, bf * 8)
        self.enc7 = ConvBlock(bf * 8, bf * 8)
        self.enc8 = ConvBlock(bf * 8, bf * 8)
        self.dec1 = DeconvBlock(bf * 8, bf * 8)
        self.dec2 = DeconvBlock(bf * 8, bf * 8)
        self.dec3 = DeconvBlock(bf * 8, bf * 8)
        self.dec4 = DeconvBlock(bf * 8, bf * 8)
        self.dec5 = DeconvBlock(bf * 8, bf * 4)
        self.dec6 = DeconvBlock(bf * 4, bf * 2)
        self.dec7 = DeconvBlock(bf * 2, bf)
        self.final = nn.ConvTranspose1d(bf, out_channels, 4, 2, 1)

    def forward(self, x):
        n = x.size(2)
        e = self.enc1(x); e = self.enc2(e); e = self.enc3(e); e = self.enc4(e)
        e = self.enc5(e); e = self.enc6(e); e = self.enc7(e); e = self.enc8(e)
        d = self.dec1(e); d = self.dec2(d); d = self.dec3(d); d = self.dec4(d)
        d = self.dec5(d); d = self.dec6(d); d = self.dec7(d)
        out = self.final(d)
        if out.size(2) != n:
            diff = n - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Paper Table II baseline: pix2pix U-Net (shared by U-Net and WGAN-GP)
# ===========================================================================
class UNetBlock(nn.Module):
    def __init__(self, i, o, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv1d(i, o, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm1d(o))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class UNetUpBlock(nn.Module):
    def __init__(self, i, o, dropout=0.0):
        super().__init__()
        layers = [nn.ConvTranspose1d(i, o, 4, 2, 1, bias=False),
                  nn.BatchNorm1d(o), nn.ReLU(inplace=True)]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip):
        x = self.block(x)
        if x.size(2) != skip.size(2):
            diff = skip.size(2) - x.size(2)
            x = F.pad(x, (diff // 2, diff - diff // 2))
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    """Paper U-Net / WGAN-GP generator -> loads CKPT['UNet'] or CKPT['WGAN-GP']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, features=64):
        super().__init__()
        f = features
        self.e1 = UNetBlock(in_channels, f, normalize=False)
        self.e2 = UNetBlock(f, f * 2); self.e3 = UNetBlock(f * 2, f * 4)
        self.e4 = UNetBlock(f * 4, f * 8); self.e5 = UNetBlock(f * 8, f * 8)
        self.e6 = UNetBlock(f * 8, f * 8); self.e7 = UNetBlock(f * 8, f * 8)
        self.e8 = UNetBlock(f * 8, f * 8, normalize=False)
        self.d1 = UNetUpBlock(f * 8, f * 8, dropout=0.5)
        self.d2 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d3 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d4 = UNetUpBlock(f * 16, f * 8)
        self.d5 = UNetUpBlock(f * 16, f * 4); self.d6 = UNetUpBlock(f * 8, f * 2)
        self.d7 = UNetUpBlock(f * 4, f)
        self.final = nn.ConvTranspose1d(f * 2, out_channels, 4, 2, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(e1); e3 = self.e3(e2); e4 = self.e4(e3)
        e5 = self.e5(e4); e6 = self.e6(e5); e7 = self.e7(e6); e8 = self.e8(e7)
        d = self.d1(e8, e7); d = self.d2(d, e6); d = self.d3(d, e5); d = self.d4(d, e4)
        d = self.d5(d, e3); d = self.d6(d, e2); d = self.d7(d, e1)
        out = self.final(d)
        if out.size(2) != x.size(2):
            diff = x.size(2) - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Proposed generator (Trail-4): U-Net + MHSA + MSDC + scenario, BatchNorm output
# ===========================================================================
class ImprovedGeneratorNoTanh(nn.Module):
    """Proposed model -> loads CKPT['Proposed']. forward(x, scenario_onehot)."""
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True))
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1), nn.BatchNorm1d(1))

    def forward(self, x, scenario_onehot):
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        e1 = self.enc1(x) + s
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Configurable generator for the R1.4 component ablation
# ===========================================================================
class ConfigurableGenerator(nn.Module):
    """Ablation generator: toggle each grouped factor independently.

    use_attention : add MHSA in the 4 bottleneck blocks
    use_dilated   : add MSDC in the 4 bottleneck blocks
    output_activation : 'batchnorm' (proposed) | 'tanh' | 'linear'
    use_scenario  : condition on the scenario one-hot (keep True to match proposed)

    Ablation rows the reviewer asked for map to:
        U-Net only          -> use_attention=False, use_dilated=False
        U-Net + MSDC        -> use_attention=False, use_dilated=True
        U-Net + MHSA        -> use_attention=True,  use_dilated=False
        U-Net + both        -> use_attention=True,  use_dilated=True
        Proposed + Tanh     -> both True, output_activation='tanh'
        Proposed + linear   -> both True, output_activation='linear'
        Proposed (BatchNorm)-> both True, output_activation='batchnorm'
    Polarity-on/off is a DATA-loader switch; adversarial-on/off is a TRAINING
    switch -- neither is an architecture change, so control them in the trainer.
    """
    def __init__(self, base_channels=64, num_scenarios=3,
                 use_attention=True, use_dilated=True,
                 output_activation="batchnorm", use_scenario=True):
        super().__init__()
        bc = base_channels
        self.use_scenario = use_scenario
        if use_scenario:
            self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(*[
            EnhancedResidualBlock(bc*8, use_attention=use_attention, use_dilated=use_dilated)
            for _ in range(4)])
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        head = [nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1)]
        if output_activation == "batchnorm":
            head.append(nn.BatchNorm1d(1))
        elif output_activation == "tanh":
            head.append(nn.Tanh())
        elif output_activation == "linear":
            pass
        else:
            raise ValueError(f"unknown output_activation {output_activation}")
        self.dec1 = nn.Sequential(*head)

    def forward(self, x, scenario_onehot=None):
        e1 = self.enc1(x)
        if self.use_scenario and scenario_onehot is not None:
            e1 = e1 + self.scenario_embed(scenario_onehot).unsqueeze(2)
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Discriminator (for R1.4 adversarial retraining)
# ===========================================================================
class SpectralNormDiscriminator(nn.Module):
    """Spectrally-normalized PatchGAN critic with scenario conditioning."""
    def __init__(self, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        sn = nn.utils.spectral_norm
        self.conv1 = sn(nn.Conv1d(2, bc, 7, 2, 3))
        self.conv2 = sn(nn.Conv1d(bc, bc*2, 5, 2, 2)); self.bn2 = nn.BatchNorm1d(bc*2)
        self.conv3 = sn(nn.Conv1d(bc*2, bc*4, 5, 2, 2)); self.bn3 = nn.BatchNorm1d(bc*4)
        self.conv4 = sn(nn.Conv1d(bc*4, bc*8, 5, 2, 2)); self.bn4 = nn.BatchNorm1d(bc*8)
        self.conv5 = sn(nn.Conv1d(bc*8, bc*8, 5, 2, 2)); self.bn5 = nn.BatchNorm1d(bc*8)
        self.pool = nn.AdaptiveAvgPool1d(1); self.fc = nn.Linear(bc*8, 1)
        self.lrelu = nn.LeakyReLU(0.2, True)

    def forward(self, pcg, ecg, scenario_onehot):
        x = torch.cat([pcg, ecg], dim=1)
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        x = self.lrelu(self.conv1(x)) + s
        x = self.lrelu(self.bn2(self.conv2(x)))
        x = self.lrelu(self.bn3(self.conv3(x)))
        x = self.lrelu(self.bn4(self.conv4(x)))
        x = self.lrelu(self.bn5(self.conv5(x)))
        return torch.sigmoid(self.fc(self.pool(x).squeeze(2)))


# ===========================================================================
# WGAN-GP critic (for retraining the WGAN-GP baseline; extracted verbatim from
# the original 05_GAN_WGAN_GP notebook). No BatchNorm, no sigmoid -> a valid
# WGAN critic. The generator is the plain UNetGenerator above.
# ===========================================================================
class WGANCritic(nn.Module):
    def __init__(self, in_channels=2, features=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(in_channels, features, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features, features * 2, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 2, features * 4, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 4, features * 8, 4, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 8, 1, 4, 1, 1),
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], dim=1))


def compute_gradient_penalty(critic, pcg, real_ecg, fake_ecg, device):
    """WGAN-GP gradient penalty. Interpolates the ECG only; PCG conditions."""
    bs = real_ecg.size(0)
    alpha = torch.rand(bs, 1, 1, device=device)
    interp = (alpha * real_ecg + (1 - alpha) * fake_ecg).requires_grad_(True)
    c_interp = critic(pcg, interp)
    grads = torch.autograd.grad(
        outputs=c_interp, inputs=interp,
        grad_outputs=torch.ones_like(c_interp),
        create_graph=True, retain_graph=True)[0]
    grads = grads.view(bs, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


# ===========================================================================
# Loading + utility helpers
# ===========================================================================
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_model(model, checkpoint_path, device="cpu", strict=False):
    """Load a checkpoint into `model`, handling common save formats."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict):
        for key in ("generator_state_dict", "model_state_dict", "state_dict"):
            if key in ckpt:
                ckpt = ckpt[key]; break
    state = {k.replace("module.", ""): v for k, v in ckpt.items()} \
            if isinstance(ckpt, dict) else ckpt
    result = model.load_state_dict(state, strict=strict)
    # Loud guard: a large key mismatch means the checkpoint does not fit this
    # architecture (e.g. a Trail-2/trial2 file loaded into the Trail-4 model).
    n_model = max(len(model.state_dict()), 1)
    n_missing, n_unexpected = len(result.missing_keys), len(result.unexpected_keys)
    tag = os.path.basename(checkpoint_path)
    if n_missing or n_unexpected:
        flag = "  <-- LIKELY WRONG CHECKPOINT FOR THIS MODEL" if n_missing > 0.10 * n_model else ""
        print(f"[load_model] {tag}: {n_missing}/{n_model} params missing, "
              f"{n_unexpected} unexpected{flag}")
    else:
        print(f"[load_model] {tag}: exact match ({n_model} tensors).")
    model.to(device).eval()
    return model, result


def build_paper_models(ckpt_dict, device="cpu"):
    """Load all four paper models. Returns {name: model}."""
    models = {}
    models["CNN"], _     = load_model(CNNEncoderDecoder(), ckpt_dict["CNN"], device)
    models["UNet"], _    = load_model(UNetGenerator(), ckpt_dict["UNet"], device)
    models["WGAN-GP"], _ = load_model(UNetGenerator(), ckpt_dict["WGAN-GP"], device)
    models["Proposed"], _= load_model(ImprovedGeneratorNoTanh(), ckpt_dict["Proposed"], device)
    return models


def run_generator(model, pcg_tensor, scenario_tensor, is_proposed):
    """Uniform inference call. `is_proposed` selects the (x, s) signature."""
    with torch.no_grad():
        if is_proposed:
            out = model(pcg_tensor, scenario_tensor)
        else:
            out = model(pcg_tensor)
    return out.squeeze().cpu().numpy()

In [ ]:
%%writefile /content/metrics.py
"""
Metrics for the JBHI revision.

Two families:
  (A) Signal-fidelity metrics  -- identical to the submitted paper's metrics.ipynb
                                  (PSNR, SNR, SSIM, Pearson r, MAE, RMSE,
                                   R-peak AMPLITUDE error, QRS-width error).
  (B) Temporal R-peak DETECTION metrics  -- NEW, added to answer Reviewer 1 #1.
       These detect R-peaks INDEPENDENTLY in the reconstructed ECG and match
       them to ground-truth peaks within a tolerance window, then report
       sensitivity, PPV, F1, FP, FN, timing-error stats, RR-interval error,
       and heart-rate error. This is the metric the paper was missing.

Keep the detector settings fixed (config_paths.RPEAK_*) so every experiment and
the paper agree.
"""

import numpy as np
import scipy.signal as signal
from scipy.stats import pearsonr


# ---------------------------------------------------------------------------
# R-peak detection (single, shared definition)
# ---------------------------------------------------------------------------
def detect_r_peaks(ecg, fs=8000, distance=400, prominence=0.5):
    """scipy find_peaks on an ECG trace. Returns integer sample indices."""
    peaks, _ = signal.find_peaks(np.asarray(ecg).ravel(),
                                 distance=distance, prominence=prominence)
    return peaks


# ---------------------------------------------------------------------------
# (A) Signal-fidelity metrics
# ---------------------------------------------------------------------------
def calc_psnr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    mse = np.mean((real - fake) ** 2)
    if mse < 1e-10:
        return 100.0
    peak = np.max(np.abs(real))
    return float(20 * np.log10(peak / np.sqrt(mse)))


def calc_snr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    noise = real - fake
    return float(10 * np.log10(np.sum(real ** 2) / (np.sum(noise ** 2) + 1e-12)))


def calc_correlation(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    try:
        c, _ = pearsonr(real, fake)
        return float(c)
    except Exception:
        return 0.0


def calc_ssim(real, fake):
    from skimage.metrics import structural_similarity as compare_ssim  # lazy: Colab-only dep
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    r = (real - real.min()) / (real.max() - real.min() + 1e-8)
    f = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(compare_ssim(r, f, data_range=1.0))


def calc_mae(real, fake):
    return float(np.mean(np.abs(np.asarray(real).ravel() - np.asarray(fake).ravel())))


def calc_rmse(real, fake):
    d = np.asarray(real).ravel() - np.asarray(fake).ravel()
    return float(np.sqrt(np.mean(d ** 2)))


def calc_r_peak_amplitude_error(real, fake, fs=8000, normalize01=True, **kw):
    """Paper's 'R-peak error' -> renamed 'R-peak AMPLITUDE error'.

    Detects R-peaks on the GROUND TRUTH only, then averages the amplitude error at
    those locations. Kept ONLY as a fidelity measure; NOT a timing metric (R1 #1).

    normalize01=True (DEFAULT) min-max normalizes both signals to [0,1] before the
    error, which REPRODUCES the paper's ~0.126 scale (confirmed: the paper's metric
    is [0,1]-normalized). normalize01=False gives the raw z-scored form, which is
    ~8x larger (z-scored ECG spans ~6-8 units vs 1 for [0,1]) -- that is what the
    first exp4/6/7 runs reported (~1.0). Use the default for consistency with the
    manuscript. Returns np.nan when the ground truth has no detectable R-peaks.
    """
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    if normalize01:
        real = (real - real.min()) / (real.max() - real.min() + 1e-8)
        fake = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(np.mean(np.abs(real[peaks] - fake[peaks])))


def _qrs_width_at_peak(ecg, peak_idx, fs=8000, threshold=0.5, search_ms=25):
    amp = abs(ecg[peak_idx]); thr = threshold * amp
    win = int(search_ms / 1000 * fs)
    q = peak_idx
    for j in range(peak_idx - 1, max(peak_idx - win, 0), -1):
        if abs(ecg[j]) < thr:
            q = j; break
    s = peak_idx
    for j in range(peak_idx + 1, min(peak_idx + win, len(ecg))):
        if abs(ecg[j]) < thr:
            s = j; break
    return (s - q) / fs * 1000.0


def calc_qrs_width_error(real, fake, fs=8000, **kw):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    errs = [abs(_qrs_width_at_peak(real, p, fs) - _qrs_width_at_peak(fake, p, fs))
            for p in peaks]
    return float(np.mean(errs))


# ---------------------------------------------------------------------------
# (B) Temporal R-peak DETECTION metrics  (NEW -- Reviewer 1 #1)
# ---------------------------------------------------------------------------
def match_peaks(true_peaks, pred_peaks, tol_samples):
    """Greedy nearest-neighbour matching within a tolerance window.

    Each true peak claims the closest unused predicted peak within tol_samples.
    Returns (matches, TP, FP, FN, signed_timing_errors_samples).
    signed error = pred_index - true_index (positive = reconstructed peak late).
    """
    true_peaks = list(np.asarray(true_peaks).astype(int))
    pred_peaks = list(np.asarray(pred_peaks).astype(int))
    used, matches, terr = set(), [], []
    for tp in true_peaks:
        cands = [(abs(pp - tp), pp) for pp in pred_peaks
                 if pp not in used and abs(pp - tp) <= tol_samples]
        if cands:
            _, pp = min(cands)
            used.add(pp); matches.append((tp, pp)); terr.append(pp - tp)
    TP = len(matches)
    FN = len(true_peaks) - TP
    FP = len(pred_peaks) - TP
    return matches, TP, FP, FN, np.asarray(terr, dtype=float)


def temporal_rpeak_metrics(true_ecg, pred_ecg, fs=8000, tol_ms=50,
                           distance=400, prominence=0.5):
    """Full temporal detection report for ONE segment.

    Returns a dict with detection (Se/PPV/F1/FP/FN), timing-error (ms),
    RR-interval error (ms), and heart-rate error (bpm). Segments with no
    ground-truth peaks return n_true=0 and NaN rates (exclude from means, but
    COUNT them -- a reconstruction that yields peaks where truth has none is a
    real failure; see caller handling).
    """
    tol = int(round(tol_ms / 1000 * fs))
    t_pk = detect_r_peaks(true_ecg, fs, distance, prominence)
    p_pk = detect_r_peaks(pred_ecg, fs, distance, prominence)

    matches, TP, FP, FN, terr = match_peaks(t_pk, p_pk, tol)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = (2 * sens * ppv / (sens + ppv)
            if (sens and ppv and not np.isnan(sens) and not np.isnan(ppv)
                and (sens + ppv) > 0) else np.nan)

    terr_ms = np.abs(terr) / fs * 1000.0
    mae_t   = float(np.mean(terr_ms)) if terr_ms.size else np.nan
    med_t   = float(np.median(terr_ms)) if terr_ms.size else np.nan
    iqr_t   = float(np.percentile(terr_ms, 75) - np.percentile(terr_ms, 25)) \
              if terr_ms.size else np.nan

    # RR-interval error on consecutive MATCHED pairs only (avoids missed-beat bias)
    rr_err = np.nan
    if len(matches) >= 2:
        tp_sorted = sorted(matches)
        rr_true, rr_pred = [], []
        for (t0, p0), (t1, p1) in zip(tp_sorted[:-1], tp_sorted[1:]):
            rr_true.append((t1 - t0) / fs * 1000.0)
            rr_pred.append((p1 - p0) / fs * 1000.0)
        rr_err = float(np.mean(np.abs(np.array(rr_true) - np.array(rr_pred))))

    # Heart-rate error (bpm), estimator = 60000 / median-RR
    def _hr(pk):
        if len(pk) < 2:
            return np.nan
        return 60000.0 / np.median(np.diff(pk) / fs * 1000.0)
    hr_true, hr_pred = _hr(t_pk), _hr(p_pk)
    hr_err = float(abs(hr_true - hr_pred)) if not (np.isnan(hr_true) or np.isnan(hr_pred)) else np.nan

    return {
        "n_true": int(len(t_pk)), "n_pred": int(len(p_pk)),
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "ppv": ppv, "f1": f1,
        "mae_timing_ms": mae_t, "median_timing_ms": med_t, "iqr_timing_ms": iqr_t,
        "rr_error_ms": rr_err,
        "hr_true_bpm": hr_true, "hr_pred_bpm": hr_pred, "hr_error_bpm": hr_err,
    }


def aggregate_temporal(rows):
    """Aggregate a list of per-segment temporal_rpeak_metrics dicts.

    Detection totals (TP/FP/FN) are POOLED then Se/PPV/F1 recomputed (the
    correct way to aggregate detection over segments). Timing/RR/HR errors are
    averaged over segments where they are defined; the valid-N is reported so
    the denominator is transparent.
    """
    rows = [r for r in rows if r is not None]
    TP = sum(r["TP"] for r in rows)
    FP = sum(r["FP"] for r in rows)
    FN = sum(r["FN"] for r in rows)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = 2 * sens * ppv / (sens + ppv) if (sens + ppv) else np.nan

    def _mean(key):
        v = [r[key] for r in rows if r[key] is not None and not np.isnan(r[key])]
        return (float(np.mean(v)), float(np.std(v)), len(v)) if v else (np.nan, np.nan, 0)

    out = {"n_segments": len(rows), "TP": TP, "FP": FP, "FN": FN,
           "sensitivity": sens, "ppv": ppv, "f1": f1}
    for k in ["mae_timing_ms", "median_timing_ms", "iqr_timing_ms",
              "rr_error_ms", "hr_error_bpm"]:
        m, s, n = _mean(k)
        out[k + "_mean"], out[k + "_std"], out[k + "_validN"] = m, s, n
    return out


# ---------------------------------------------------------------------------
# Preprocessing helpers (shared with the paper pipeline)
# ---------------------------------------------------------------------------
def ensure_positive_polarity(ecg):
    """Flip so R-peaks point up. Applied to the ECG TARGET only (never PCG)."""
    ecg = np.asarray(ecg)
    return -ecg if np.abs(ecg.min()) > np.abs(ecg.max()) else ecg


def normalize_zscore(sig):
    sig = np.asarray(sig, dtype=np.float32)
    return ((sig - sig.mean()) / (sig.std() + 1e-8)).astype(np.float32)

In [ ]:
%%writefile /content/ieee_style.py
"""
IEEE double-column figure style for the JBHI revision.

Design goals (per IEEE Transactions / J-BHI figure rules):
  * Vector output (PDF) + high-DPI PNG (600) for every figure.
  * Serif face (Times) to match IEEE body text; DejaVu Serif fallback on Colab.
  * BOLD axis labels that always carry explicit units.
  * Thick spines and ticks so lines stay legible at column scale.
  * constrained_layout everywhere -> no text/label/legend overlaps.
  * Column widths fixed to IEEE spec: single = 3.5 in, double = 7.16 in.

Usage
-----
    from ieee_style import set_ieee_style, new_fig, finalize_axes, save_fig
    set_ieee_style()
    fig, ax = new_fig("single", height=2.2)
    ax.plot(t, y)
    finalize_axes(ax, xlabel="Time (s)", ylabel="Amplitude (a.u.)")
    save_fig(fig, out_dir, "fig_name")   # writes fig_name.pdf AND fig_name.png
"""

import matplotlib
matplotlib.use("Agg")            # headless-safe; remove if you want inline display
import matplotlib.pyplot as plt

# IEEE column widths (inches)
COL_SINGLE = 3.5
COL_DOUBLE = 7.16

# Colour-blind-safe palette (Okabe-Ito), stable role -> colour mapping
COLORS = {
    "truth":    "#009E73",   # bluish green  -> ground truth
    "proposed": "#0072B2",   # blue          -> proposed model
    "cnn":      "#D55E00",   # vermillion    -> CNN
    "unet":     "#CC79A7",   # reddish purple-> U-Net
    "wgan":     "#E69F00",   # orange        -> WGAN-GP
    "pcg":      "#666666",   # grey          -> PCG input
    "accent":   "#0072B2",
    "muted":    "#999999",
}


def set_ieee_style():
    """Apply the global rcParams. Call once at the top of every script."""
    plt.rcParams.update({
        "figure.dpi":        150,
        "savefig.dpi":       600,
        "savefig.bbox":      "tight",
        "savefig.pad_inches": 0.02,
        "figure.constrained_layout.use": True,
        "font.size":         8,
        "font.family":       "serif",
        "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset":  "dejavuserif",
        "text.usetex":       False,
        "axes.labelsize":    9,
        "axes.labelweight":  "bold",
        "axes.titlesize":    9,
        "axes.titleweight":  "bold",
        "axes.linewidth":    1.1,
        "axes.grid":         True,
        "grid.alpha":        0.25,
        "grid.linewidth":    0.4,
        "xtick.labelsize":   8,
        "ytick.labelsize":   8,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size":  3.5,
        "ytick.major.size":  3.5,
        "xtick.direction":   "out",
        "ytick.direction":   "out",
        "legend.fontsize":   7.5,
        "legend.framealpha": 0.92,
        "legend.edgecolor":  "0.7",
        "legend.handlelength": 1.6,
        "lines.linewidth":   1.0,
        "lines.solid_capstyle": "round",
    })


def new_fig(width="single", height=2.2, nrows=1, ncols=1, **kw):
    """Create a figure sized to an IEEE column.

    width : 'single' (3.5 in), 'double' (7.16 in), or a float in inches.
    height: figure height in inches.
    Returns (fig, ax) or (fig, axes) exactly like plt.subplots.
    """
    w = {"single": COL_SINGLE, "double": COL_DOUBLE}.get(width, width)
    fig, ax = plt.subplots(nrows, ncols, figsize=(w, height), **kw)
    return fig, ax


def finalize_axes(ax, xlabel=None, ylabel=None, title=None,
                  legend=False, legend_loc="best", legend_ncol=1):
    """Apply bold labels with units, thicken spines, tidy ticks.

    Always pass xlabel/ylabel WITH units, e.g. 'Time (ms)', 'PSNR (dB)'.
    """
    if xlabel:
        ax.set_xlabel(xlabel, fontweight="bold")
    if ylabel:
        ax.set_ylabel(ylabel, fontweight="bold")
    if title:
        ax.set_title(title, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)
    ax.tick_params(axis="both", which="major", width=1.0, labelsize=8)
    if legend:
        leg = ax.legend(loc=legend_loc, ncol=legend_ncol, frameon=True)
        leg.get_frame().set_linewidth(0.6)
    return ax


def save_fig(fig, out_directory, name):
    """Save both a vector PDF (for LaTeX) and a 600-dpi PNG (for preview)."""
    import os
    pdf = os.path.join(out_directory, f"{name}.pdf")
    png = os.path.join(out_directory, f"{name}.png")
    fig.savefig(pdf)
    fig.savefig(png)
    plt.close(fig)
    print(f"  saved  {pdf}")
    print(f"  saved  {png}")
    return pdf, png

In [ ]:
import importlib
import config_paths, models, metrics, ieee_style
for m in (config_paths, models, metrics, ieee_style):
    importlib.reload(m)
ieee_style.set_ieee_style()
print('shared modules ready | device =', config_paths.DEVICE)

## Experiment

# exp4 - Blind cross-dataset PSNR (Reviewer 1, comment 8)

The submitted paper claims that "amplitude reconstruction transfers" across
datasets because SensSmart-Tech PSNR (17.04 dB) nearly matches the EPHNOGRAM
PSNR (17.10 dB). That match is an artifact. The cross-dataset pipeline rescales
every generator output by the target standard deviation:

    fake_ecg = fake_raw * (real_std / fake_std)

`real_std` is measured on the GROUND-TRUTH ECG, so the rescaling injects target
information into the prediction. The honest, blind number uses the raw generator
output with no target-std rescaling. This script computes that blind number and
reports it next to the rescaled 17.04 dB.

The script also runs four supporting checks:
  1. mean std ratio (fake_std / real_std ~ 0.28), the size of the rescaling gap;
  2. a no-bandpass control (training used no digital bandpass), which re-runs the
     blind PSNR without the 25-400 Hz filter to test the filtering confound;
  3. WFDB header verification (base_time, base_date, fs) for the paired _ecg and
     _pcg records, evidence for or against sample-level synchronization;
  4. a scale-invariance proof that SSIM and Pearson r are unchanged by the
     rescaling, so the correlation collapse is genuine, not an amplitude effect.

Outputs: a JSON results file, a paste-ready LaTeX table (metric, EPHNOGRAM,
SmartTech-rescaled, SmartTech-blind), and one IEEE figure (Welch PSD of the two
datasets' PCG on a single axis, the device-response cause of the collapse).

In [ ]:
# --- Standard agent header (this cell is dropped by build_notebooks.py, which
#     mounts Drive and writes the shared modules for you). Kept so the file also
#     runs as a stand-alone Colab notebook.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import sys
    sys.path.append('/content/drive/MyDrive/JBHI_Revision/shared')  # copy shared/ here first
    print("NOTE: this stand-alone header expects the local shared/ folder copied to")
    print("      /content/drive/MyDrive/JBHI_Revision/shared  (or paste the four")
    print("      modules into /content). build_notebooks.py does this automatically.")
except Exception as _e:
    print(f"Not on Colab or Drive already mounted ({_e}); assuming shared modules on path.")

import config_paths, models, metrics, ieee_style

In [ ]:
# --- Experiment imports, config, and output folder ---
import os
import json
import datetime
import numpy as np
import torch
import scipy.signal as ss

# wfdb is Colab-only; install on demand so the header verification and no-filter
# control can read the raw SensSmart-Tech records.
try:
    import wfdb
except ImportError:
    import subprocess
    print("Installing wfdb ...")
    subprocess.check_call(["pip", "install", "-q", "wfdb"])
    import wfdb

ieee_style.set_ieee_style()

EXP_NAME = "exp4_R1_8_blind_crossdataset"
OUT      = config_paths.out_dir(EXP_NAME)
DEVICE   = config_paths.DEVICE

# Re-run from scratch instead of loading cached stage results.
FORCE = False
# Optional caps (None = use every record). Keep None for the paper numbers.
MAX_CONTROL_RECORDS = None    # WFDB no-bandpass control
MAX_PSD_SEGMENTS    = 40      # segments per dataset averaged into the Welch PSD

# EPHNOGRAM reference column (paper Table / VERIFIED locally). Recomputed below
# if the EPHNOGRAM chunks are present; otherwise these constants are used.
EPH_REF = {
    "psnr": 17.10, "ssim": 0.903, "correlation": 0.610, "rpeak_amp_err": 0.126,
    "source": "paper reported / verified",
}

print(f"Device      : {DEVICE}")
print(f"Output dir  : {OUT}")
print(f"WFDB dir    : {config_paths.SENSSMART_WFDB}")
print(f"Cached npz  : {config_paths.SENSSMART_PREP}")


def _save_json(obj, name):
    path = os.path.join(OUT, name)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    print(f"  saved  {path}")
    return path


def _cache_failed(d):
    """True when a cached stage holds no records, or holds records whose every
    metric is NaN. The July 14 run cached 338 records with validN 0 for every
    metric, from the unstable 8 kHz ECG bandpass. A record count alone does not
    show that a stage succeeded, so test the aggregates for a single valid value.
    """
    if not isinstance(d, dict):
        return True
    counts = [v["validN"] for v in d.values()
              if isinstance(v, dict) and "validN" in v]
    for agg in (v for v in d.values() if isinstance(v, dict)):
        counts += [v["validN"] for v in agg.values()
                   if isinstance(v, dict) and "validN" in v]
    return not counts or max(counts) == 0


def _load_json(name):
    path = os.path.join(OUT, name)
    if (not FORCE) and os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return None


def _agg(values):
    """Mean/std/min/max/median over finite values, with the valid-N denominator."""
    v = np.asarray([x for x in values if x is not None and np.isfinite(x)], dtype=float)
    if v.size == 0:
        return {"mean": None, "std": None, "min": None, "max": None,
                "median": None, "validN": 0}
    return {"mean": float(v.mean()), "std": float(v.std()),
            "min": float(v.min()), "max": float(v.max()),
            "median": float(np.median(v)), "validN": int(v.size)}

## Load the proposed generator

The proposed model (`ImprovedGeneratorNoTanh`, 19.32M parameters) is loaded from
the Trail-4 checkpoint. Inference conditions on the exercise scenario one-hot,
matching the cross-dataset protocol in the paper.

In [ ]:
generator = None
GEN_PARAMS = None
ckpt_path = config_paths.CKPT["Proposed"]
if not os.path.exists(ckpt_path):
    ckpt_path = config_paths.CKPT_PROPOSED_ALT

if os.path.exists(ckpt_path):
    generator = models.ImprovedGeneratorNoTanh()
    generator, missing = models.load_model(generator, ckpt_path, device=DEVICE)
    GEN_PARAMS = models.count_params(generator)
    print(f"Generator loaded from {ckpt_path}")
    print(f"  trainable parameters: {GEN_PARAMS:,}")
    if missing is not None and (getattr(missing, 'missing_keys', None) or
                                getattr(missing, 'unexpected_keys', None)):
        print(f"  load_state_dict report: {missing}")
else:
    print(f"CHECKPOINT NOT FOUND at {ckpt_path}. Inference stages will be skipped.")

SCEN = torch.tensor(config_paths.SCEN_EXERCISE, dtype=torch.float32).unsqueeze(0).to(DEVICE)


def reconstruct(pcg):
    """Run the generator on one z-scored PCG segment. Returns raw output (no rescale)."""
    x = torch.from_numpy(np.asarray(pcg, dtype=np.float32)).unsqueeze(0).unsqueeze(0).to(DEVICE)
    return models.run_generator(generator, x, SCEN, is_proposed=True)

## Helpers: WFDB loading and preprocessing

The main filtered numbers come from the cached, already-preprocessed npz files
(identical to the pipeline that produced the paper's 17.04 dB). The no-bandpass
control and the header verification read the raw WFDB records directly, so the
only difference between the two blind numbers is the 25-400 Hz filter.

In [ ]:
def list_wfdb_records():
    """Record bases that have BOTH an _ecg and a _pcg record (the valid pairs).

    SensSmart-Tech stores each modality as a SEPARATE WFDB record named
    {subject}_{timestamp}_{ecg|pcg|ppg|acc}. Only sessions that have both an _ecg
    and a _pcg record form an ECG-PCG pair. Requiring both avoids the
    'No such file: ..._pcg.hea' failures seen on a partially-uploaded folder.
    The full PhysioNet database has 338 such pairs.
    """
    d = config_paths.SENSSMART_WFDB
    if not os.path.isdir(d):
        return []
    files = set(os.listdir(d))
    ecg = {f[:-len("_ecg.hea")] for f in files if f.endswith("_ecg.hea")}
    pcg = {f[:-len("_pcg.hea")] for f in files if f.endswith("_pcg.hea")}
    return sorted(ecg & pcg)


def _resample_to_target(sig, fs_in, fs_out=config_paths.FS):
    if int(fs_in) == int(fs_out):
        return np.asarray(sig, dtype=np.float32)
    g = np.gcd(int(fs_out), int(fs_in))
    return ss.resample_poly(sig, int(fs_out) // g, int(fs_in) // g).astype(np.float32)


def _bandpass(sig, fs, band, order=config_paths.FILTER_ORDER):
    # SOS + sosfiltfilt is numerically stable at low normalized cutoffs, where the
    # (b, a) form with filtfilt blows up to NaN (e.g. 0.5 Hz at 8 kHz -> cutoff
    # 0.000125). Guard the band against the Nyquist limit as well.
    nyq = fs / 2.0
    hi = min(band[1], nyq * 0.99)
    sos = ss.butter(order, [band[0] / nyq, hi / nyq], btype="band", output="sos")
    return ss.sosfiltfilt(sos, sig).astype(np.float32)


def read_wfdb_pair(base):
    """Load one raw ECG (lead II) + PCG pair. Returns (ecg, pcg, fs) or (None, None, None)."""
    try:
        d = config_paths.SENSSMART_WFDB
        ecg_rec = wfdb.rdrecord(os.path.join(d, f"{base}_ecg"))
        pcg_rec = wfdb.rdrecord(os.path.join(d, f"{base}_pcg"))
        names = list(ecg_rec.sig_name)
        lead = names.index("II") if "II" in names else 1
        ecg = ecg_rec.p_signal[:, lead].astype(np.float32)
        pcg = pcg_rec.p_signal.squeeze().astype(np.float32)
        return ecg, pcg, int(ecg_rec.fs)
    except Exception as e:
        print(f"  read failed for {base}: {e}")
        return None, None, None


def preprocess_pair(ecg_raw, pcg_raw, fs_in, apply_bandpass):
    """Resample -> optional bandpass -> ECG polarity fix -> z-score -> truncate.

    apply_bandpass=False reproduces the training condition (no digital bandpass).
    Returns (ecg, pcg) each of length SEG_LEN, or (None, None) when too short.
    """
    # Filter at the NATIVE rate (fs_in), BEFORE resampling. At 1 kHz the ECG
    # 0.5-40 Hz band is well conditioned; at 8 kHz it is not (all-NaN).
    if apply_bandpass:
        ecg_raw = _bandpass(ecg_raw, fs_in, config_paths.ECG_BAND)
        pcg_raw = _bandpass(pcg_raw, fs_in, config_paths.PCG_BAND)
    ecg = _resample_to_target(ecg_raw, fs_in)
    pcg = _resample_to_target(pcg_raw, fs_in)
    ecg = metrics.normalize_zscore(metrics.ensure_positive_polarity(ecg))
    pcg = metrics.normalize_zscore(pcg)
    T = config_paths.SEG_LEN
    if len(ecg) < T or len(pcg) < T:
        return None, None
    return ecg[:T], pcg[:T]

## Stage 1 - Blind vs rescaled on the cached SmartTech records

For each cached record, the script computes the raw (blind) reconstruction and
the target-std-rescaled reconstruction, then scores both. PSNR separates the two
columns; SSIM and Pearson r stay identical because they are scale-invariant.

In [ ]:
def score_pair(real, fake_blind):
    """Return blind and rescaled metrics for one segment, plus the std ratio."""
    real_std = float(np.std(real))
    fake_std = float(np.std(fake_blind))
    ratio = fake_std / (real_std + 1e-8)                 # ~0.28 cross-dataset
    fake_rescaled = fake_blind * (real_std / (fake_std + 1e-8))
    return {
        "psnr_blind":     metrics.calc_psnr(real, fake_blind),
        "psnr_rescaled":  metrics.calc_psnr(real, fake_rescaled),
        "ssim_blind":     metrics.calc_ssim(real, fake_blind),
        "ssim_rescaled":  metrics.calc_ssim(real, fake_rescaled),
        "corr_blind":     metrics.calc_correlation(real, fake_blind),
        "corr_rescaled":  metrics.calc_correlation(real, fake_rescaled),
        "rpeak_blind":    metrics.calc_r_peak_amplitude_error(real, fake_blind, fs=config_paths.FS),
        "rpeak_rescaled": metrics.calc_r_peak_amplitude_error(real, fake_rescaled, fs=config_paths.FS),
        "std_ratio":      ratio,
    }


stage1 = _load_json("stage1_filtered.json")
if stage1 is not None and _cache_failed(stage1):
    print("Stage 1: cached result is empty or all-NaN -> discarding it and recomputing.")
    stage1 = None
stage1_recomputed = stage1 is None
if stage1 is None and generator is not None:
    prep = config_paths.SENSSMART_PREP
    use_cache = os.path.isdir(prep) and any(f.endswith(".npz") for f in os.listdir(prep))
    rows = []
    if use_cache:
        npz_files = sorted(f for f in os.listdir(prep) if f.endswith(".npz"))
        print(f"Stage 1: scoring {len(npz_files)} cached SmartTech records ...")
        for i, fn in enumerate(npz_files):
            data = np.load(os.path.join(prep, fn))          # already preprocessed
            real = metrics.normalize_zscore(metrics.ensure_positive_polarity(data["ecg"]))[:config_paths.SEG_LEN]
            pcg = metrics.normalize_zscore(data["pcg"])[:config_paths.SEG_LEN]
            if len(real) < config_paths.SEG_LEN or len(pcg) < config_paths.SEG_LEN:
                continue
            m = score_pair(real, reconstruct(pcg)); m["record"] = fn[:-4]; rows.append(m)
            if (i + 1) % 50 == 0:
                print(f"  {i + 1}/{len(npz_files)}")
    else:
        # No cache -> read the raw WFDB pairs directly (the full 338-pair database).
        bases = list_wfdb_records()
        print(f"Stage 1: no npz cache; reading {len(bases)} raw WFDB ECG-PCG pairs ...")
        for i, base in enumerate(bases):
            ecg_raw, pcg_raw, fs_in = read_wfdb_pair(base)
            if ecg_raw is None:
                continue
            real, pcg = preprocess_pair(ecg_raw, pcg_raw, fs_in, apply_bandpass=True)
            if real is None:
                continue
            m = score_pair(real, reconstruct(pcg)); m["record"] = base; rows.append(m)
            if (i + 1) % 25 == 0:
                print(f"  {i + 1}/{len(bases)}  (kept {len(rows)})")

    keys = ["psnr_blind", "psnr_rescaled", "ssim_blind", "ssim_rescaled",
            "corr_blind", "corr_rescaled", "rpeak_blind", "rpeak_rescaled", "std_ratio"]
    stage1 = {"n_records": len(rows),
              "agg": {k: _agg([r[k] for r in rows]) for k in keys},
              "per_record": rows}
    _save_json(stage1, "stage1_filtered.json")
elif stage1 is not None:
    print(f"Stage 1: loaded cached results ({stage1['n_records']} records).")
else:
    print("Stage 1 skipped (generator missing).")

def _fmt(x, spec=".2f"):
    return format(x, spec) if isinstance(x, (int, float)) else "n/a"

if stage1 is not None and stage1.get("n_records", 0) > 0:
    a = stage1["agg"]
    print("\n  SmartTech (filtered):")
    print(f"    PSNR blind    : {_fmt(a['psnr_blind']['mean'])} dB   (validN {a['psnr_blind']['validN']})")
    print(f"    PSNR rescaled : {_fmt(a['psnr_rescaled']['mean'])} dB   (paper headline 17.04)")
    print(f"    SSIM blind/rescaled : {_fmt(a['ssim_blind']['mean'], '.3f')} / {_fmt(a['ssim_rescaled']['mean'], '.3f')}")
    print(f"    Corr blind/rescaled : {_fmt(a['corr_blind']['mean'], '.3f')} / {_fmt(a['corr_rescaled']['mean'], '.3f')}")
    print(f"    mean std ratio : {_fmt(a['std_ratio']['mean'], '.3f')}   (expected ~0.28)")
else:
    print("\n  SmartTech: 0 records scored. Upload the full WFDB (338 ECG-PCG pairs)")
    print("  to the shared drive, delete the old exp4 stage*.json, and re-run.")

## Stage 2 - No-bandpass control (filtering confound)

Training applied no digital bandpass. This control re-reads the raw WFDB records
and computes the blind PSNR with and without the 25-400 Hz filter on the same
records, so any gap is attributable to the filter alone.

In [ ]:
stage2 = _load_json("stage2_bandpass_control.json")
if stage2 is not None and _cache_failed(stage2):
    print("Stage 2: cached result is empty or all-NaN -> discarding it and recomputing.")
    stage2 = None
if stage2 is None and generator is not None:
    bases = list_wfdb_records()
    if MAX_CONTROL_RECORDS is not None:
        bases = bases[:MAX_CONTROL_RECORDS]
    print(f"Stage 2: no-bandpass control over {len(bases)} WFDB records ...")
    filt_blind, nofilt_blind = [], []
    for i, base in enumerate(bases):
        ecg_raw, pcg_raw, fs_in = read_wfdb_pair(base)
        if ecg_raw is None:
            continue
        ef, pf = preprocess_pair(ecg_raw, pcg_raw, fs_in, apply_bandpass=True)
        en, pn = preprocess_pair(ecg_raw, pcg_raw, fs_in, apply_bandpass=False)
        if ef is not None:
            filt_blind.append(metrics.calc_psnr(ef, reconstruct(pf)))
        if en is not None:
            nofilt_blind.append(metrics.calc_psnr(en, reconstruct(pn)))
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(bases)}")

    stage2 = {
        "n_filtered":   len(filt_blind),
        "n_nofilter":   len(nofilt_blind),
        "psnr_blind_filtered":  _agg(filt_blind),
        "psnr_blind_nofilter":  _agg(nofilt_blind),
    }
    _save_json(stage2, "stage2_bandpass_control.json")
elif stage2 is not None:
    print("Stage 2: loaded cached control results.")
else:
    print("Stage 2 skipped (generator missing).")

if stage2 is not None and stage2["psnr_blind_filtered"]["mean"] is not None:
    f_m = stage2["psnr_blind_filtered"]["mean"]
    n_m = stage2["psnr_blind_nofilter"]["mean"]
    print(f"\n  Blind PSNR with bandpass    : {f_m:.2f} dB (n={stage2['n_filtered']})")
    print(f"  Blind PSNR without bandpass : {n_m:.2f} dB (n={stage2['n_nofilter']})")
    print(f"  Filtering shifts blind PSNR by {n_m - f_m:+.2f} dB.")

## Stage 3 - Header verification (synchronization evidence)

The script reads the WFDB headers of the paired _ecg and _pcg records and checks
whether they share a base date, base time, and sampling rate. Matching timestamps
are evidence of a common clock; missing or mismatched timestamps mean sample-level
alignment is not guaranteed, which explains the correlation collapse.

In [ ]:
stage3 = _load_json("stage3_sync_headers.json")
if stage3 is None:
    bases = list_wfdb_records()
    print(f"Stage 3: reading headers for {len(bases)} record pairs ...")
    records, n_time_present, n_time_match, n_fs_match = [], 0, 0, 0
    examples = []
    for base in bases:
        try:
            he = wfdb.rdheader(os.path.join(config_paths.SENSSMART_WFDB, f"{base}_ecg"))
            hp = wfdb.rdheader(os.path.join(config_paths.SENSSMART_WFDB, f"{base}_pcg"))
        except Exception as e:
            examples.append({"record": base, "error": str(e)})
            continue

        def _s(x):
            return None if x is None else str(x)

        et, ed, pt, pd = _s(he.base_time), _s(he.base_date), _s(hp.base_time), _s(hp.base_date)
        fs_match = int(he.fs) == int(hp.fs)
        time_present = (et is not None) and (pt is not None)
        time_match = time_present and (et == pt) and (ed == pd)
        n_fs_match += int(fs_match)
        n_time_present += int(time_present)
        n_time_match += int(time_match)

        offset_s = None
        if time_present:
            try:
                fmt = "%H:%M:%S"
                t_e = datetime.datetime.strptime(et.split(".")[0], fmt)
                t_p = datetime.datetime.strptime(pt.split(".")[0], fmt)
                offset_s = (t_p - t_e).total_seconds()
            except Exception:
                offset_s = None

        rec = {"record": base, "ecg_base_date": ed, "ecg_base_time": et,
               "pcg_base_date": pd, "pcg_base_time": pt,
               "ecg_fs": int(he.fs), "pcg_fs": int(hp.fs),
               "fs_match": fs_match, "time_present": time_present,
               "time_match": time_match, "ecg_pcg_time_offset_s": offset_s}
        records.append(rec)
        if len(examples) < 5:
            examples.append(rec)

    n = len(records)
    stage3 = {
        "n_pairs": n,
        "n_fs_match": n_fs_match,
        "n_base_time_present": n_time_present,
        "n_ecg_pcg_time_match": n_time_match,
        "frac_time_present": (n_time_present / n) if n else None,
        "frac_time_match": (n_time_match / n) if n else None,
        "examples": examples,
        "records": records,
    }
    _save_json(stage3, "stage3_sync_headers.json")
else:
    print("Stage 3: loaded cached header results.")

if stage3 is not None and stage3["n_pairs"]:
    print(f"\n  Header pairs read      : {stage3['n_pairs']}")
    print(f"  sampling-rate matches  : {stage3['n_fs_match']}/{stage3['n_pairs']}")
    print(f"  base_time present      : {stage3['n_base_time_present']}/{stage3['n_pairs']}")
    print(f"  ecg/pcg time match     : {stage3['n_ecg_pcg_time_match']}/{stage3['n_pairs']}")

## Stage 4 - Scale-invariance proof

SSIM min-max normalizes both signals and Pearson r divides out amplitude, so a
positive rescaling cannot change either. Stage 1 already stored blind and rescaled
values for both; this stage measures the maximum per-record difference (expected
~0) and contrasts it with the PSNR gap the rescaling opens.

In [ ]:
stage4 = _load_json("stage4_scale_invariance.json")
if stage4 is not None and stage1_recomputed:
    print("Stage 4: Stage 1 was recomputed -> discarding the derived cache.")
    stage4 = None            # stage 4 reads stage1["per_record"] and cannot outlive it
if stage4 is None and stage1 is not None:
    rows = stage1["per_record"]
    ssim_diffs = [abs(r["ssim_blind"] - r["ssim_rescaled"]) for r in rows
                  if r["ssim_blind"] is not None and r["ssim_rescaled"] is not None]
    corr_diffs = [abs(r["corr_blind"] - r["corr_rescaled"]) for r in rows
                  if r["corr_blind"] is not None and r["corr_rescaled"] is not None]
    psnr_gaps = [r["psnr_rescaled"] - r["psnr_blind"] for r in rows
                 if r["psnr_blind"] is not None and r["psnr_rescaled"] is not None]

    # Independent synthetic sanity check: scale a segment by 5x and confirm equality.
    rng = np.random.default_rng(0)
    real_d = rng.standard_normal(4000).astype(np.float32)
    fake_d = (0.28 * rng.standard_normal(4000)).astype(np.float32)
    synth = {
        "ssim_1x_vs_5x_diff": abs(metrics.calc_ssim(real_d, fake_d) -
                                  metrics.calc_ssim(real_d, 5.0 * fake_d)),
        "corr_1x_vs_5x_diff": abs(metrics.calc_correlation(real_d, fake_d) -
                                  metrics.calc_correlation(real_d, 5.0 * fake_d)),
    }
    stage4 = {
        "max_ssim_abs_diff": float(np.max(ssim_diffs)) if ssim_diffs else None,
        "max_corr_abs_diff": float(np.max(corr_diffs)) if corr_diffs else None,
        "mean_psnr_gap_db":  float(np.mean(psnr_gaps)) if psnr_gaps else None,
        "synthetic_check":   synth,
    }
    _save_json(stage4, "stage4_scale_invariance.json")
elif stage4 is not None:
    print("Stage 4: loaded cached scale-invariance results.")
else:
    print("Stage 4 skipped (Stage 1 results missing).")

if stage4 is not None:
    print(f"\n  max |SSIM blind - rescaled| : {stage4['max_ssim_abs_diff']}")
    print(f"  max |corr blind - rescaled| : {stage4['max_corr_abs_diff']}")
    print(f"  mean PSNR gap (rescaled-blind): {stage4['mean_psnr_gap_db']} dB")
    print(f"  synthetic 1x-vs-5x diffs      : {stage4['synthetic_check']}")

## Stage 5 (optional) - Recompute the EPHNOGRAM reference

When the EPHNOGRAM test chunks are present, the script recomputes the in-domain
blind and rescaled metrics. In domain, the std ratio is near 1, so blind and
rescaled PSNR nearly coincide, unlike the cross-dataset case. When the chunks are
absent, the constants in `EPH_REF` are used.

In [ ]:
def _scen_onehot(idx):
    v = [0.0, 0.0, 0.0]
    v[idx] = 1.0
    return v


stage5 = _load_json("stage5_ephnogram.json")
if stage5 is None and generator is not None and os.path.isdir(config_paths.EPH_CHUNKS_DIR):
    test_subjects = None
    if os.path.exists(config_paths.SPLIT_FILE):
        with open(config_paths.SPLIT_FILE, encoding="utf-8") as f:
            split = json.load(f)
        test_subjects = set(split.get("test_subjects", split.get("test", [])))
    files = sorted(f for f in os.listdir(config_paths.EPH_CHUNKS_DIR)
                   if f.endswith(".npz") and "split" not in f)
    print(f"Stage 5: recomputing EPHNOGRAM reference over up to {len(files)} chunks ...")
    rows = []
    for fn in files:
        try:
            data = np.load(os.path.join(config_paths.EPH_CHUNKS_DIR, fn), allow_pickle=True)
        except Exception:
            continue
        if "ecg" not in data or "pcg" not in data:
            continue
        subj = str(np.asarray(data["subject_id"]).ravel()[0]) if "subject_id" in data else ""
        if test_subjects and subj not in test_subjects:
            continue
        real = metrics.normalize_zscore(metrics.ensure_positive_polarity(data["ecg"]))[:config_paths.SEG_LEN]
        pcg  = metrics.normalize_zscore(data["pcg"])[:config_paths.SEG_LEN]
        if len(real) < config_paths.SEG_LEN or len(pcg) < config_paths.SEG_LEN:
            continue
        scen_idx = config_paths.scenario_to_index(np.asarray(data["scenario"]).ravel()[0]) \
            if "scenario" in data else 1
        x = torch.from_numpy(pcg.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(DEVICE)
        s = torch.tensor(_scen_onehot(scen_idx), dtype=torch.float32).unsqueeze(0).to(DEVICE)
        fake = models.run_generator(generator, x, s, is_proposed=True)
        rows.append(score_pair(real, fake))

    if rows:
        keys = ["psnr_blind", "psnr_rescaled", "ssim_blind", "ssim_rescaled",
                "corr_blind", "corr_rescaled", "rpeak_blind", "rpeak_rescaled", "std_ratio"]
        stage5 = {"n_segments": len(rows),
                  "agg": {k: _agg([r[k] for r in rows]) for k in keys}}
        _save_json(stage5, "stage5_ephnogram.json")
        print(f"  EPHNOGRAM recomputed over {len(rows)} segments.")
    else:
        print("  No matching EPHNOGRAM chunks; keeping EPH_REF constants.")
elif stage5 is not None:
    print("Stage 5: loaded cached EPHNOGRAM recompute.")
else:
    print("Stage 5 skipped (generator or EPHNOGRAM chunks missing); using EPH_REF constants.")

# Resolve the EPHNOGRAM reference column: recomputed rescaled values if available.
if stage5 is not None and stage5.get("agg"):
    a = stage5["agg"]
    EPH_COL = {
        "psnr": a["psnr_rescaled"]["mean"], "ssim": a["ssim_rescaled"]["mean"],
        "correlation": a["corr_rescaled"]["mean"], "rpeak_amp_err": a["rpeak_rescaled"]["mean"],
        "psnr_blind": a["psnr_blind"]["mean"], "std_ratio": a["std_ratio"]["mean"],
        "source": "recomputed",
    }
else:
    EPH_COL = dict(EPH_REF)
    EPH_COL["psnr_blind"] = None
    EPH_COL["std_ratio"] = None

## Assemble the results JSON and the LaTeX table

The table columns are EPHNOGRAM, SmartTech (rescaled), and SmartTech (blind).
PSNR is the only row that moves between the two SmartTech columns; SSIM and
Pearson r are identical, which is exactly the point of R1.8.

In [ ]:
def _fmt(x, nd=2):
    return "n/a" if x is None else f"{x:.{nd}f}"


results = {
    "experiment": EXP_NAME,
    "generator_params": GEN_PARAMS,
    "ephnogram_reference": EPH_COL,
    "smarttech_filtered": (stage1["agg"] if stage1 else None),
    "smarttech_n": (stage1["n_records"] if stage1 else None),
    "bandpass_control": stage2,
    "synchronization": (None if stage3 is None else
                        {k: stage3[k] for k in
                         ["n_pairs", "n_fs_match", "n_base_time_present",
                          "n_ecg_pcg_time_match", "frac_time_present", "frac_time_match"]}),
    "scale_invariance": stage4,
}
_save_json(results, "exp4_results.json")

# Pull the SmartTech column values (fall back to the paper headline when uncomputed).
if stage1 is not None:
    st = stage1["agg"]
    st_col = {
        "psnr_rescaled": st["psnr_rescaled"]["mean"], "psnr_blind": st["psnr_blind"]["mean"],
        "ssim_rescaled": st["ssim_rescaled"]["mean"], "ssim_blind": st["ssim_blind"]["mean"],
        "corr_rescaled": st["corr_rescaled"]["mean"], "corr_blind": st["corr_blind"]["mean"],
        "rpeak_rescaled": st["rpeak_rescaled"]["mean"], "rpeak_blind": st["rpeak_blind"]["mean"],
        "std_ratio": st["std_ratio"]["mean"], "n": stage1["n_records"],
    }
else:
    st_col = {"psnr_rescaled": 17.04, "psnr_blind": None, "ssim_rescaled": 0.742,
              "ssim_blind": None, "corr_rescaled": -0.003, "corr_blind": None,
              "rpeak_rescaled": 1.400, "rpeak_blind": None, "std_ratio": 0.28, "n": 310}

eph = EPH_COL
n_st = st_col["n"]
ratio_txt = _fmt(st_col["std_ratio"], 2)

latex = r"""% ---------------------------------------------------------------------------
% exp4 (R1.8): blind cross-dataset fidelity.
% Paste into the manuscript near the cross-dataset paragraph and cite as
% Table~\ref{tab:blind_crossdataset}. Replace the prior "amplitude transfers"
% sentence with the honest reading in the caption.
% ---------------------------------------------------------------------------
\begin{table}[!t]
\centering
\caption{Cross-dataset signal fidelity on SensSmart-Tech (%N records), evaluated
blind and after target-standard-deviation rescaling. The rescaled PSNR uses the
ground-truth standard deviation, so it overstates transfer. The blind PSNR is the
honest number. SSIM and Pearson $r$ are scale-invariant and therefore identical in
both columns, so the correlation collapse reflects temporal misalignment, not
amplitude. The mean std ratio is %RATIO.}
\label{tab:blind_crossdataset}
\begin{tabular}{@{}lccc@{}}
\toprule
Metric & EPHNOGRAM & SmartTech (rescaled) & SmartTech (blind) \\
\midrule
PSNR (dB)        & %EP_PSNR & %ST_PSNR_R & %ST_PSNR_B \\
SSIM             & %EP_SSIM & %ST_SSIM_R & %ST_SSIM_B \\
Pearson $r$      & %EP_CORR & %ST_CORR_R & %ST_CORR_B \\
R-peak amp. err. & %EP_RPK  & %ST_RPK_R  & %ST_RPK_B \\
\bottomrule
\end{tabular}
\end{table}
"""

repl = {
    "%N": str(n_st), "%RATIO": ratio_txt,
    "%EP_PSNR": _fmt(eph["psnr"], 2), "%EP_SSIM": _fmt(eph["ssim"], 3),
    "%EP_CORR": _fmt(eph["correlation"], 3), "%EP_RPK": _fmt(eph["rpeak_amp_err"], 3),
    "%ST_PSNR_R": _fmt(st_col["psnr_rescaled"], 2), "%ST_PSNR_B": _fmt(st_col["psnr_blind"], 2),
    "%ST_SSIM_R": _fmt(st_col["ssim_rescaled"], 3), "%ST_SSIM_B": _fmt(st_col["ssim_blind"], 3),
    "%ST_CORR_R": _fmt(st_col["corr_rescaled"], 3), "%ST_CORR_B": _fmt(st_col["corr_blind"], 3),
    "%ST_RPK_R": _fmt(st_col["rpeak_rescaled"], 3), "%ST_RPK_B": _fmt(st_col["rpeak_blind"], 3),
}
for k, v in repl.items():
    latex = latex.replace(k, v)

tex_path = os.path.join(OUT, "table_blind_crossdataset.tex")
with open(tex_path, "w", encoding="utf-8") as f:
    f.write(latex)
print(f"  saved  {tex_path}\n")
print(latex)

## IEEE figure - Welch PSD of SmartTech vs EPHNOGRAM PCG

One axis, two averaged spectra. Different sensor front ends give different PCG
frequency responses; that device-response mismatch is the physical cause of the
correlation collapse. Axis labels are bold and carry units.

In [ ]:
def _avg_welch(pcg_segments, fs=config_paths.FS, nperseg=8192):
    acc, cnt = None, 0
    for seg in pcg_segments:
        f, pxx = ss.welch(np.asarray(seg, dtype=np.float64), fs=fs, nperseg=nperseg)
        acc = pxx if acc is None else acc + pxx
        cnt += 1
    if cnt == 0:
        return None, None
    return f, acc / cnt


# Collect unfiltered, z-scored PCG from each dataset (unfiltered so the native
# device response is visible; both training pipelines used no digital bandpass).
smart_pcg = []
for base in list_wfdb_records()[:MAX_PSD_SEGMENTS]:
    ecg_raw, pcg_raw, fs_in = read_wfdb_pair(base)
    if pcg_raw is None:
        continue
    _, p = preprocess_pair(ecg_raw, pcg_raw, fs_in, apply_bandpass=False)
    if p is not None:
        smart_pcg.append(p)

eph_pcg = []
if os.path.isdir(config_paths.EPH_CHUNKS_DIR):
    ef = sorted(f for f in os.listdir(config_paths.EPH_CHUNKS_DIR)
                if f.endswith(".npz") and "split" not in f)[:MAX_PSD_SEGMENTS]
    for fn in ef:
        try:
            d = np.load(os.path.join(config_paths.EPH_CHUNKS_DIR, fn), allow_pickle=True)
        except Exception:
            continue
        if "pcg" in d:
            eph_pcg.append(metrics.normalize_zscore(d["pcg"])[:config_paths.SEG_LEN])

f_s, p_s = _avg_welch(smart_pcg) if smart_pcg else (None, None)
f_e, p_e = _avg_welch(eph_pcg) if eph_pcg else (None, None)

if p_s is not None or p_e is not None:
    fig, ax = ieee_style.new_fig("double", height=2.8)
    if p_e is not None:
        ax.plot(f_e, 10.0 * np.log10(p_e + 1e-20),
                color=ieee_style.COLORS["proposed"], linewidth=1.0,
                label=f"EPHNOGRAM PCG (n={len(eph_pcg)})")
    if p_s is not None:
        ax.plot(f_s, 10.0 * np.log10(p_s + 1e-20),
                color=ieee_style.COLORS["cnn"], linewidth=1.0,
                label=f"SensSmart-Tech PCG (n={len(smart_pcg)})")
    ax.set_xlim(0, 1000)
    ieee_style.finalize_axes(ax, xlabel="Frequency (Hz)", ylabel="PSD (dB/Hz)",
                             legend=True, legend_loc="upper right")
    ieee_style.save_fig(fig, OUT, "fig_pcg_psd_device_response")
else:
    print("Figure skipped: no PCG segments available for either dataset.")

print("\nexp4 complete. Artifacts in:", OUT)